# Module 3.5 — Enums & Typed Named Tuples

### Python Mastery · Track 3: Object-Oriented Programming & Design Patterns

When a value can only be one of a fixed set of options, an enumeration gives those options clear names instead of bare strings or numbers. Typed named tuples provide lightweight, immutable records with named, type-annotated fields. This module covers both, which make code safer and more readable.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it.
- Compare the enum approach with using plain strings, and see the safety it adds.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Define an enumeration with `enum.Enum` and use its members.
2. Use `auto()` for automatic values and iterate over members.
3. Use `IntEnum` where integer behaviour is needed.
4. Combine options with `Flag`.
5. Build typed, immutable records with `typing.NamedTuple`.

**Prerequisites:** Tracks 1 and 2, and Modules 3.1 to 3.4.

---

## Part 1 · Why Enums, and the Basics

Representing a fixed set of choices as plain strings (`"red"`, `"green"`) or numbers invites typos and gives no protection: nothing stops a misspelt `"gren"`. An **enumeration** defines named constants that group together, compare safely, and read clearly.

Define one by subclassing `enum.Enum`. Each member has a `name` and a `value`.

In [ ]:
from enum import Enum

class Color(Enum):
    RED = 1
    GREEN = 2
    BLUE = 3

print("a member:", Color.RED)
print("its name:", Color.RED.name, "| its value:", Color.RED.value)

# Members are singletons, compared by identity, which is fast and safe.
print("identity comparison:", Color.RED is Color.RED)
print("equality:", Color.RED == Color.GREEN)

# Look up a member by value or by name.
print("by value:", Color(2))
print("by name:", Color["BLUE"])

## Part 2 · `auto()` and Iterating Members

When the specific values do not matter, `auto()` assigns them for you. Enums are iterable, so you can list every option, and membership can be tested. This makes enums convenient for menus, states, and any closed set of choices.

In [ ]:
from enum import Enum, auto

class Direction(Enum):
    NORTH = auto()       # values are assigned automatically: 1, 2, 3, 4
    EAST = auto()
    SOUTH = auto()
    WEST = auto()

# Iterate over all members in definition order.
for d in Direction:
    print(d.name, "=", d.value)

print("number of directions:", len(Direction))
print("is NORTH a member:", Direction.NORTH in Direction)

Enums are well suited to representing state. Using them in comparisons reads clearly and prevents invalid values from slipping in.

In [ ]:
from enum import Enum, auto

class TrafficLight(Enum):
    RED = auto()
    YELLOW = auto()
    GREEN = auto()

def action(light):
    if light is TrafficLight.RED:
        return "stop"
    elif light is TrafficLight.YELLOW:
        return "slow down"
    elif light is TrafficLight.GREEN:
        return "go"

for light in TrafficLight:
    print(light.name, "->", action(light))

## Part 3 · `IntEnum` for Integer Behaviour

A plain `Enum` member is not an integer and will not compare with numbers. When you need members that also behave as integers, for example to compare priority levels or to interoperate with code expecting numbers, use `IntEnum`.

In [ ]:
from enum import IntEnum

class Priority(IntEnum):
    LOW = 1
    MEDIUM = 2
    HIGH = 3
    CRITICAL = 4

# IntEnum members compare and sort like integers.
print("HIGH > LOW:", Priority.HIGH > Priority.LOW)
print("sorted:", sorted([Priority.CRITICAL, Priority.LOW, Priority.MEDIUM]))
print("arithmetic works:", Priority.HIGH + 1)

# Still readable when printed.
print("name retained:", Priority.HIGH.name)

## Part 4 · `Flag` for Combinable Options

Some options are meant to be combined, such as a set of permissions. `Flag` (and `IntFlag`) supports this with the bitwise operators `|` (combine), `&` (test), and `~`. Each member should have a power-of-two value so combinations stay distinct.

In [ ]:
from enum import Flag, auto

class Permission(Flag):
    READ = auto()        # auto() gives powers of two for Flag: 1, 2, 4
    WRITE = auto()
    EXECUTE = auto()

# Combine permissions with |.
access = Permission.READ | Permission.WRITE
print("combined:", access)

# Test membership with the 'in' operator (Python 3.11+) or with &.
print("has READ:", Permission.READ in access)
print("has EXECUTE:", Permission.EXECUTE in access)
print("test with &:", bool(access & Permission.WRITE))

# Add and remove flags.
full = access | Permission.EXECUTE
print("with execute added:", full)
print("write removed:", full & ~Permission.WRITE)

## Part 5 · Typed Named Tuples

Module 2.4 introduced `collections.namedtuple`. The `typing.NamedTuple` form adds **type annotations** and a cleaner class syntax, producing an immutable record with named fields. It is excellent for small, fixed data structures where immutability and clarity matter, and it supports default values and methods.

In [ ]:
from typing import NamedTuple

class Point(NamedTuple):
    x: int
    y: int
    label: str = "origin"        # a default value

    def distance_from_origin(self):
        return (self.x ** 2 + self.y ** 2) ** 0.5

p = Point(3, 4)
print("named access:", p.x, p.y, p.label)
print("a method on the record:", p.distance_from_origin())

# It is still a tuple: indexing, unpacking, and immutability all apply.
print("indexing:", p[0])
x, y, label = p
print("unpacked:", x, y, label)

# Immutable: p.x = 10 would raise an error. Use _replace for a changed copy.
print("changed copy:", p._replace(x=10))

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — A basic enum (Easy)

In [ ]:
from enum import Enum

class Weekday(Enum):
    MONDAY = 1
    TUESDAY = 2
    WEDNESDAY = 3

day = Weekday.TUESDAY
print(day.name, "is day number", day.value)
# Experiment: look up Weekday(3) and Weekday["MONDAY"].

### Example 2 — A typed named tuple (Easy)

In [ ]:
from typing import NamedTuple

class Book(NamedTuple):
    title: str
    pages: int

b = Book("Dune", 412)
print(f"{b.title} has {b.pages} pages")

### Example 3 — Iterating an enum to build a menu (Medium)

In [ ]:
from enum import Enum, auto

class Course(Enum):
    STARTER = auto()
    MAIN = auto()
    DESSERT = auto()

print("Menu:")
for i, course in enumerate(Course, start=1):
    print(f"  {i}. {course.name.title()}")

### Example 4 — Priorities with IntEnum (Medium)

In [ ]:
from enum import IntEnum

class Severity(IntEnum):
    INFO = 1
    WARNING = 2
    ERROR = 3

incidents = [Severity.WARNING, Severity.ERROR, Severity.INFO, Severity.ERROR]

# Because they behave as integers, we can sort and find the maximum directly.
print("most severe:", max(incidents).name)
print("sorted:", [s.name for s in sorted(incidents)])
print("errors and worse:", [s.name for s in incidents if s >= Severity.ERROR])

### Example 5 — A permission system with Flag (Difficult)

In [ ]:
from enum import Flag, auto

class Access(Flag):
    NONE = 0
    READ = auto()
    WRITE = auto()
    DELETE = auto()
    ADMIN = READ | WRITE | DELETE      # a composite defined from others

roles = {
    "viewer": Access.READ,
    "editor": Access.READ | Access.WRITE,
    "admin": Access.ADMIN,
}

def can(role, permission):
    return permission in roles[role]

for role in roles:
    print(f"{role:7} can delete: {can(role, Access.DELETE)}, can read: {can(role, Access.READ)}")

### Example 6 — Typed records with methods and defaults (Difficult)

In [ ]:
from typing import NamedTuple

class Money(NamedTuple):
    amount: float
    currency: str = "USD"

    def __str__(self):
        return f"{self.amount:.2f} {self.currency}"

    def converted(self, rate, to):
        """Return a new Money in another currency."""
        return Money(round(self.amount * rate, 2), to)

price = Money(100.0)
print("original:", price)
print("converted:", price.converted(0.92, "EUR"))

# Records compare by value and can be sorted, being tuples underneath.
basket = [Money(50), Money(20), Money(75)]
print("cheapest:", str(min(basket, key=lambda m: m.amount)))

---

## Exercises

**Exercise 1 (Easy).** Define an enum `Suit` with members `HEARTS`, `DIAMONDS`, `CLUBS`, `SPADES` using `auto()`. Print each member's name and value by iterating.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Define a typed named tuple `Coordinate` with `lat` and `lon` (both float). Create one and print its fields.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Define an `IntEnum` called `Level` with `BRONZE = 1`, `SILVER = 2`, `GOLD = 3`. Given a list of levels, print the highest one's name and sort the list ascending.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Using a `Flag` enum `Feature` with `DARK_MODE`, `NOTIFICATIONS`, and `BETA`, combine two of them and write a check that reports whether `BETA` is enabled in the combination.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Define a typed named tuple `Employee` with `name` (str), `salary` (float), and `department` (str, default `"general"`), plus a method `raised(percent)` that returns a new `Employee` with an increased salary. Demonstrate the raise.

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
from enum import Enum, auto

class Suit(Enum):
    HEARTS = auto()
    DIAMONDS = auto()
    CLUBS = auto()
    SPADES = auto()

for s in Suit:
    print(s.name, s.value)

**Solution 2**

In [ ]:
from typing import NamedTuple

class Coordinate(NamedTuple):
    lat: float
    lon: float

c = Coordinate(18.5, 73.8)
print(c.lat, c.lon)

**Solution 3**

In [ ]:
from enum import IntEnum

class Level(IntEnum):
    BRONZE = 1
    SILVER = 2
    GOLD = 3

levels = [Level.SILVER, Level.GOLD, Level.BRONZE, Level.SILVER]
print("highest:", max(levels).name)
print("sorted:", [l.name for l in sorted(levels)])

**Solution 4**

In [ ]:
from enum import Flag, auto

class Feature(Flag):
    DARK_MODE = auto()
    NOTIFICATIONS = auto()
    BETA = auto()

active = Feature.DARK_MODE | Feature.BETA
print("beta enabled:", Feature.BETA in active)

**Solution 5**

In [ ]:
from typing import NamedTuple

class Employee(NamedTuple):
    name: str
    salary: float
    department: str = "general"

    def raised(self, percent):
        return self._replace(salary=round(self.salary * (1 + percent / 100), 2))

e = Employee("Asha", 50000)
print("before:", e)
print("after raise:", e.raised(10))

---

## Summary and Key Points

- An enum (`enum.Enum`) names a fixed set of options; members have a `name` and `value`, are singletons compared by identity, and can be looked up by value or name.
- `auto()` assigns values automatically; enums are iterable and support membership tests, which suits states and menus.
- `IntEnum` members also behave as integers, so they compare, sort, and do arithmetic while keeping readable names.
- `Flag` (and `IntFlag`) members combine with `|`, are tested with `in` or `&`, and removed with `& ~`; give members power-of-two values.
- `typing.NamedTuple` builds immutable, type-annotated records with named fields, defaults, and methods, while remaining a tuple (indexable, unpackable, comparable).

### Next module: 3.6 — Design Patterns in Python

The next module surveys common design patterns, creational, structural, and behavioural, and shows how Python's dynamic features often simplify them.